# PATHDEFENSE dataset preprocesssing
Code to prepare data for experiments in "Defense Against Shortest Path Attacks"

This material is based upon work supported by the United States Air Force under
Air  Force  Contract  No.  FA8702-15-D-0001  and  the  Combat  Capabilities
Development Command Army Research Laboratory (under Cooperative Agreement Number
W911NF-13-2-0045).  Any  opinions,  findings,  conclusions  or  recommendations
expressed in this material are those of the authors and do not necessarily
reflect the views of the United States Air Force or Army Research Laboratory.

Copyright (C) 2023
Benjamin A. Miller, Zohair Shafi, Wheeler Ruml, Yevgeniy Vorobeychik, Tina Eliassi-Rad, and Scott Alfeld

The software is provided to you on an As-Is basis

Delivered to the U.S. Government with Unlimited Rights, as defined in DFARS Part
252.227-7013 or 7014 (Feb 2014). Notwithstanding any copyright notice, U.S.
Government rights in this work are defined by DFARS 252.227-7013 or DFARS
252.227-7014 as detailed above. Use of this work other than as specifically
authorized by the U.S. Government may violate any copyrights that exist in this
work

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import pickle as pkl


### UK Metro data

In [ ]:
df = pd.read_csv('Data_Release_v1.11/edges.csv')
G1 = nx.from_pandas_edgelist(df[(df['ori_layer']==3) & (df['des_layer']==3)], source='ori_node', target='des_node', edge_attr='minutes', create_using=nx.DiGraph)
G1.remove_edges_from(nx.selfloop_edges(G1))
lcc = list(max(nx.strongly_connected_components(G1), key=len))
G1 = nx.subgraph(G1, lcc)

In [ ]:
G = nx.Graph()
for e in G1.edges:
    if e[0] < e[1]: 
        if G1.has_edge(e[1], e[0]):
            w = G1.edges[e]['minutes'] + G1.edges[(e[1], e[0])]['minutes']
            w /= 2
            G.add_edge(e[0], e[1], weight=w)
        else:
            print('no return')
            w = G1.edges[e]['minutes']
            G.add_edge(e[0], e[1], weight=w)
    elif not G1.has_edge(e[1], e[0]):
        print('no return')
        w = G1.edges[e]['minutes']
        G.add_edge(e[0], e[1], weight=w)

In [ ]:
with open('metro.pkl', 'wb') as f:
    pkl.dump(G, f)

### Autonomous System data

In [ ]:
df = pd.read_csv('as19991211.txt', sep='\t', skiprows=4, header=None)
G = nx.from_pandas_edgelist(df, source=0, target=1)
A = nx.adjacency_matrix(G)
A.setdiag(np.zeros((G.number_of_nodes())))
A.eliminate_zeros()
G = nx.from_scipy_sparse_matrix(A)
with open('as19991211.pkl', 'wb') as f:
    pkl.dump(G, f)

### Hypertext '09 data

In [ ]:
df = pd.read_csv('ht09_contact_list.dat', sep='\t', header=None)
G = nx.from_pandas_edgelist(df, source=1, target=2, create_using=nx.MultiGraph)
A = nx.adjacency_matrix(G)
A.setdiag(np.zeros((G.number_of_nodes())))
A.eliminate_zeros()
G = nx.from_scipy_sparse_matrix(A)
for e in G.edges:
    G.edges[e]['weight'] = np.ceil(42-4*np.log2(G.edges[e]['weight']))
with open('ht09.pkl', 'wb') as f:
    pkl.dump(G, f)

### US Airport data

In [ ]:
df = pd.read_csv('USairport500.txt', sep=' ', header=None)
G = nx.from_pandas_edgelist(df, source=0, target=1, edge_attr=2)
A = nx.adjacency_matrix(G, weight=2)
A.setdiag(np.zeros((G.number_of_nodes())))
A.eliminate_zeros()
G = nx.from_scipy_sparse_matrix(A)
for e in G.edges:
    G.edges[e]['weight'] = np.ceil(43-2*np.log2(G.edges[e]['weight']))
with open('airport.pkl', 'wb') as f:
    pkl.dump(G, f)